In [2]:
import numpy as np
import pandas as pd

raw = "../data-raw"
LEAGUE = "GB1"          # a variable so you can switch leagues later
FIRST_SEASON = 2010     # valuation coverage is dense from around here

players = pd.read_csv(f"{raw}/players.csv.gz", parse_dates=["date_of_birth"])
vals = pd.read_csv(f"{raw}/player_valuations.csv.gz", parse_dates=["date"])
games = pd.read_csv(f"{raw}/games.csv.gz", parse_dates=["date"])
apps = pd.read_csv(f"{raw}/appearances.csv.gz", parse_dates=["date"])

league_games = games.loc[games["competition_id"] == LEAGUE, ["game_id", "season"]]
apps_l = apps.merge(league_games, on="game_id", how="inner")
apps_l = apps_l[apps_l["season"] >= FIRST_SEASON]

season_stats = (
    apps_l.groupby(["player_id", "season"])
    .agg(
        appearances=("game_id", "nunique"),
        minutes=("minutes_played", "sum"),
        goals=("goals", "sum"),
        assists=("assists", "sum"),
        yellows=("yellow_cards", "sum"),
        reds=("red_cards", "sum"),
    )
    .reset_index()
)

for col in ["goals", "assists"]:
    season_stats[f"{col}_per90"] = season_stats[col] / season_stats["minutes"].clip(lower=1) * 90

season_stats["snapshot_date"] = pd.to_datetime((season_stats["season"] + 1).astype(str) + "-06-30")

season_stats = season_stats.sort_values("snapshot_date")
vals_sorted = vals.sort_values("date")

data = pd.merge_asof(
    season_stats, vals_sorted[["player_id", "date", "market_value_in_eur"]],
    left_on="snapshot_date", right_on="date", by="player_id",
    direction="backward", tolerance=pd.Timedelta(days=180),
)

data = data.merge(players[["player_id", "date_of_birth", "position", "sub_position"]], on="player_id", how="left")
data["age"] = (data["snapshot_date"] - data["date_of_birth"]).dt.days / 365.25

data = data.dropna(subset=["market_value_in_eur"])
data = data[data["market_value_in_eur"] > 0]
data["log_value"] = np.log1p(data["market_value_in_eur"])


print(data.shape)
print("duplicates (player, season):", data.duplicated(["player_id", "season"]).sum())   # must be 0
print(data.groupby("season").size())     # about 500-600 per season is expected
print(data.isna().mean())                # anything unexpected?
print(data["age"].describe())            # roughly 16-40

data.to_csv("../data-processed/player_seasons.csv", index=False)

(7114, 18)
duplicates (player, season): 0
season
2012    491
2013    507
2014    483
2015    512
2016    506
2017    500
2018    477
2019    493
2020    518
2021    505
2022    526
2023    542
2024    543
2025    511
dtype: int64
player_id              0.000000
season                 0.000000
appearances            0.000000
minutes                0.000000
goals                  0.000000
assists                0.000000
yellows                0.000000
reds                   0.000000
goals_per90            0.000000
assists_per90          0.000000
snapshot_date          0.000000
date                   0.000000
market_value_in_eur    0.000000
date_of_birth          0.000562
position               0.000000
sub_position           0.000000
age                    0.000562
log_value              0.000000
dtype: float64
count    7110.000000
mean       27.195019
std         4.160390
min        16.495551
25%        24.175222
50%        27.121150
75%        30.003422
max        43.118412
Name: age, 